# Gaussian Speckle Noise 1024 synthetic-image example

Multiplicative Gaussian speckle noise. This notebook starts from the same synthetic 1024 x 1024 image, applies one AGFB noise model, and displays the clean image, noisy image, and residual.


## Noise Context

Speckle noise is a multiplicative granular pattern caused by coherent wave interference. It is common in synthetic-aperture radar, ultrasound, laser imaging, optical coherence tomography, and other systems where coherent illumination or wave scattering is imaged. This notebook uses a Gaussian multiplicative approximation for quick stress testing.

Goodman's paper [Some Fundamental Properties of Speckle](https://doi.org/10.1364/JOSA.66.001145) discusses coherent speckle statistics. A general background reference is [Speckle (interference)](https://en.wikipedia.org/wiki/Speckle_%28interference%29).


## Setup

Import the model function and notebook helpers. The autoreload lines help Jupyter pick up local source edits without restarting the kernel.


In [ ]:
# Imports and local reloads.
%load_ext autoreload
%autoreload 2

import time

import torch

from agfb_noise import add_speckle
from agfb_noise.helpers.notebook import show_noise_preview, summarize_tensors, synthetic_1024_image

## Synthetic Image

Create the shared 1024 x 1024 synthetic input on CUDA when available.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed = 123

clean = synthetic_1024_image(device=device)
clean.shape, clean.device, clean.dtype

## Apply Noise

Set the model parameters and corrupt the image.


In [ ]:
def apply_model() -> torch.Tensor:
    return add_speckle(clean, sigma=0.18, seed=seed, clamp=(0.0, 1.0))


noisy = apply_model()
delta = noisy - clean

## Summary

Report compact tensor statistics for the clean, noisy, and residual images.


In [ ]:
summarize_tensors(
    {
        "clean": clean,
        "noisy": noisy,
        "delta": delta,
    }
)

## Preview

The preview down-samples the 1024 image for display. The residual uses a blue-white-garnet diverging map.


In [ ]:
show_noise_preview(clean, noisy, title="Gaussian Speckle Noise")

## Timing

Time one hot-path model application on the selected device.


In [ ]:
if clean.is_cuda:
    torch.cuda.synchronize()
start = time.perf_counter()
_ = apply_model()
if clean.is_cuda:
    torch.cuda.synchronize()
elapsed_ms = (time.perf_counter() - start) * 1000.0
elapsed_ms